In [ ]:
dataset_url = ""
dataset_base64 = ""
num_classes = 2
epochs = 5
img_size = 64

In [ ]:
import json, io, base64, warnings, os
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import numpy as np

result = {}
try:
    # ── Check required deps ────────────────────────────────────────────────────
    try:
        from PIL import Image
        import tensorflow as tf
        tf.get_logger().setLevel('ERROR')
    except ImportError as e:
        raise ImportError(f"Missing dependency: {e}. Run: pip install pillow tensorflow")

    img_sz = int(img_size)
    n_cls = int(num_classes)
    n_epochs = int(epochs)

    # ── Image loaders ──────────────────────────────────────────────────────────
    def load_from_zip(zip_bytes):
        import zipfile
        zf = zipfile.ZipFile(io.BytesIO(zip_bytes))
        images, labels, class_names = [], [], []
        for name in sorted(zf.namelist()):
            low = name.lower()
            if not any(low.endswith(ext) for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']):
                continue
            parts = [p for p in name.replace('\\', '/').split('/') if p and not p.startswith('.')]
            if len(parts) < 2:
                continue
            cls = parts[-2]
            if cls not in class_names:
                class_names.append(cls)
            lbl = class_names.index(cls)
            try:
                img = Image.open(io.BytesIO(zf.read(name))).convert('RGB')
                img = img.resize((img_sz, img_sz))
                images.append(np.array(img, dtype=np.float32) / 255.0)
                labels.append(lbl)
            except Exception:
                continue
        return np.array(images), np.array(labels), class_names

    # ── Load or generate data ──────────────────────────────────────────────────
    raw_data = None
    if dataset_url:
        import urllib.request
        with urllib.request.urlopen(dataset_url, timeout=30) as r:
            raw_data = r.read()
    elif dataset_base64:
        raw_data = base64.b64decode(dataset_base64)

    X, y, class_names = None, None, None
    if raw_data:
        try:
            X, y, class_names = load_from_zip(raw_data)
        except Exception:
            pass

    if X is None or len(X) == 0:
        # Synthetic demo dataset
        np.random.seed(42)
        n_demo = max(40, n_cls * 20)
        X = np.random.rand(n_demo, img_sz, img_sz, 3).astype(np.float32)
        y = np.repeat(np.arange(n_cls), n_demo // n_cls)[:n_demo]
        class_names = [f'class_{i}' for i in range(n_cls)]

    actual_cls = max(n_cls, len(class_names))
    class_names = (class_names + [f'class_{i}' for i in range(len(class_names), actual_cls)])[:actual_cls]

    # ── Build CNN ──────────────────────────────────────────────────────────────
    split = int(len(X) * 0.8)
    X_train, X_val = X[:split], X[split:]
    y_train, y_val = y[:split], y[split:]

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(img_sz, img_sz, 3)),
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(2, 2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(actual_cls, activation='softmax'),
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    hist_obj = model.fit(X_train, y_train,
                         epochs=n_epochs,
                         validation_data=(X_val, y_val),
                         verbose=0,
                         batch_size=16)
    h = hist_obj.history

    training_history = [
        {
            'epoch':        i + 1,
            'accuracy':     round(float(h['accuracy'][i]), 4),
            'val_accuracy': round(float(h['val_accuracy'][i]), 4),
            'loss':         round(float(h['loss'][i]), 4),
            'val_loss':     round(float(h['val_loss'][i]), 4),
        }
        for i in range(len(h['accuracy']))
    ]

    n_s = min(10, len(X_val))
    preds = model.predict(X_val[:n_s], verbose=0)
    pred_cls = np.argmax(preds, axis=1)
    sample_preds = [
        {
            'image_index': i,
            'actual':      class_names[int(y_val[i])] if int(y_val[i]) < len(class_names) else str(int(y_val[i])),
            'predicted':   class_names[int(pred_cls[i])] if int(pred_cls[i]) < len(class_names) else str(int(pred_cls[i])),
            'confidence':  round(float(preds[i][pred_cls[i]]), 4),
        }
        for i in range(n_s)
    ]

    result = {
        'algorithm': 'cnn',
        'num_classes': actual_cls,
        'metrics': {
            'accuracy':     round(float(h['accuracy'][-1]), 4),
            'val_accuracy': round(float(h['val_accuracy'][-1]), 4),
            'loss':         round(float(h['loss'][-1]), 4),
            'val_loss':     round(float(h['val_loss'][-1]), 4),
        },
        'training_history': training_history,
        'class_names': class_names,
        'sample_predictions': sample_preds,
    }

except Exception as e:
    import traceback
    result = {'error': str(e), 'traceback': traceback.format_exc()}

with open('/tmp/ml_result.json', 'w') as f:
    json.dump(result, f)
print(json.dumps(result, indent=2))